In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pandas as pd
from imlff.analysis.base import fetch_runs_df
from schnetpack.md.data import HDF5Loader
from imlff.common.util import proj_path    
from imlff.data.info import molecule_to_n_atoms

In [ ]:
save_name = 'mds/2026-06-25_iter-count'
project_name = 'md-v7-test'
tags = ['june23']

pat = proj_path('result_summary', save_name+'.csv')
if os.path.exists(pat) and 0:
    sims_df_new = pd.read_csv(pat)
else:
    kwargs = {}
    if tags:
        kwargs['tags'] = {
            '$in': tags,
        }
    sims_df_new = fetch_runs_df(
        project_name=project_name,
        max_runs=None,
        **kwargs,
        fix=False
    )
    
    for col in ['model_config', 'molecule_config', 'md_config', 'train']:
        sims_df_new = pd.concat([
            sims_df_new[col].apply(pd.Series).add_prefix(f"{col}__"),
            sims_df_new.drop([col], axis=1),
        ], axis=1)
    
    
    os.makedirs(os.path.dirname(pat), exist_ok=True)
    sims_df_new.to_csv(pat)
    

if 'sims_df_full' in locals():
    sims_df_full = pd.concat([sims_df_full, sims_df_new], ignore_index=True)
else:
    sims_df_full = sims_df_new

sims_df_full['n_atoms'] = sims_df_full.train__molecule.apply(molecule_to_n_atoms)
sims_df_full = sims_df_full.sort_values('_timestamp')

Fetching runs from md-sab-parallel-scan with filters {'tags': {'$in': ['june23']}}
Found 68 runs.
after filter: 68


100%|███████████████████████████████████████████████████████████████████████████████| 68/68 [00:00<00:00, 28687.66it/s]


68

## Loading hdf5s for exact iteration counts

In [ ]:
# Initialize the column if it doesn't exist
if 'data' not in sims_df_full.columns:
    sims_df_full['avg_iterations'] = None

for i, row in sims_df_full.sort_values('_step').iterrows():
    # Skip if data is not present / too big
    if not pd.isna(row.data) and row.data is not None:
        print(f"Skipping {row.train__molecule} - not loaded")
        continue
    if row._step > 200_000:
        print(f"Skipping {row.train__molecule}. Steps:", row._step)
        continue
    
    
    print(f"Loading {row.train__molecule}. Steps:", row._step)
    
    log_file = proj_path(row.proj_path)
    assert os.path.exists(f"{log_file}.hdf5")
    
    with HDF5Loader(f"{log_file}.hdf5", skip_initial=0) as data:
    
        fi = data.get_property('forw_iterations', atomistic=False)
        avg_iterations = (fi + bi).mean() / 2
        
        # Write data directly into the DataFrame
        sims_df_full.at[i, 'avg_iterations'] = avg_iterations
    

pat_with_stats = pat.replace('.csv', '_with-stats.csv')
sims_df_full.to_csv(pat_with_stats)
print(pat_with_stats)

~/imlff/result_summary/mds/2026-06-25_iter-count_with-stats.csv
